In [1]:
# ============================================================
# GOVERNANCE SECTION GENERATOR
# IFRS S1/S2 governance | Azure OpenAI REST endpoint
# ============================================================

import os
import json
import re
import urllib.request
import urllib.error
from typing import TypedDict, Literal
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langgraph.graph import StateGraph, START, END

# ── ENV LOADING ──────────────────────────────────────────────
# In notebooks, .env is often not loaded because the kernel runs from a different folder.
# find_dotenv() makes the notebook search upward from the current path.
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_URL = os.getenv("AZURE_OPENAI_CHAT_URL")

if AZURE_OPENAI_CHAT_URL:
    AZURE_OPENAI_CHAT_URL = AZURE_OPENAI_CHAT_URL.strip().strip('"').strip("'")

if not AZURE_OPENAI_API_KEY:
    raise ValueError("Missing AZURE_OPENAI_API_KEY. Add it to .env or environment variables.")

if not AZURE_OPENAI_CHAT_URL:
    raise ValueError(
        "Missing AZURE_OPENAI_CHAT_URL. Expected full Azure URL, e.g.\n"
        "https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=2024-02-01"
    )

if not AZURE_OPENAI_CHAT_URL.startswith("https://"):
    raise ValueError(f"Invalid Azure URL: {AZURE_OPENAI_CHAT_URL!r}")

print("Azure OpenAI REST config loaded")
print("URL loaded:", AZURE_OPENAI_CHAT_URL[:80] + "...")
print("Key loaded:", True, "| length:", len(AZURE_OPENAI_API_KEY))


def _azure_chat_completion(messages: list[dict], temperature: float, max_tokens: int, json_mode: bool = False) -> dict:
    """Minimal Azure OpenAI REST call using the full URL stored in AZURE_OPENAI_CHAT_URL."""
    payload = {
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if json_mode:
        payload["response_format"] = {"type": "json_object"}

    req = urllib.request.Request(
        AZURE_OPENAI_CHAT_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "api-key": AZURE_OPENAI_API_KEY,
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode(errors="replace")
        print("Azure HTTP error:", e.code)
        print(body[:1500])
        raise
    except urllib.error.URLError as e:
        print("Azure URL / connection error:", e)
        print("URL used:", repr(AZURE_OPENAI_CHAT_URL))
        raise


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    data = _azure_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=2200,
        json_mode=False,
    )
    return data["choices"][0]["message"]["content"]


def call_llm_json(system_prompt: str, user_prompt: str) -> dict:
    data = _azure_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=1600,
        json_mode=True,
    )
    return json.loads(data["choices"][0]["message"]["content"])

print("LLM helper functions ready")


Loaded .env from: c:\Users\BV426BP\Documents\IFRS Data\.env
Azure OpenAI REST config loaded
URL loaded: https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-4o-...
Key loaded: True | length: 32
LLM helper functions ready


In [2]:
# ── LOAD PAYLOAD ─────────────────────────────────────────────
# Works both locally and in this sandbox if the payload is placed next to the notebook.

candidate_paths = [
    Path("Data/payload_BANK01.json"),
    Path("payload_BANK01.json"),
    Path.cwd() / "Data" / "payload_BANK01.json",
    Path.cwd() / "payload_BANK01.json",
]

PAYLOAD_PATH = next((p for p in candidate_paths if p.exists()), None)
if PAYLOAD_PATH is None:
    raise FileNotFoundError(
        "Could not find payload_BANK01.json. Put it in Data/payload_BANK01.json "
        "or in the same folder as the notebook."
    )

with open(PAYLOAD_PATH, "r", encoding="utf-8") as f:
    payload = json.load(f)

bank_name = payload["bank"]["bank_name"]
print(f"Loaded payload for: {bank_name}")
print(f"Payload path: {PAYLOAD_PATH}")


Loaded payload for: Eurolux Universal Bank AG
Payload path: payload_BANK01.json


In [3]:
# ── EVIDENCE EXTRACTOR ───────────────────────────────────────
# Pulls only the governance-relevant fields from the payload.
# Strict governance fixes added:
# - formal mandate / charter evidence is separated from activity evidence
# - board trade-off evidence is explicitly checked; if absent, the writer must state that limitation
# - management process evidence is converted into a process flow, not only an inventory
# - skills adequacy process evidence is separated from skills outcome metrics
# - assurance scope limitation is made explicit for financed emissions / Scope 3
# - meeting_id retained for board decision traceability


def _is_present(value) -> bool:
    return value is not None and str(value).strip().lower() not in {"", "nan", "none", "null"}


def _safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def _normalise_text(value) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def extract_management_process_evidence(payload: dict, year: int = 2024) -> dict:
    """Summarise the process evidence needed for management responsibility."""
    risks = [
        r for r in payload.get("climate_risk_register", [])
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
    ]

    if not risks:
        return {
            "risk_register_available": False,
            "message": "No climate_risk_register records available for the reporting year.",
            "process_flow_instruction": (
                "Do not invent management process details. State that the available evidence does not "
                "include risk-register process records for the reporting year."
            )
        }

    frequencies = sorted({str(r.get("monitoring_frequency")) for r in risks if _is_present(r.get("monitoring_frequency"))})
    risk_categories = sorted({str(r.get("risk_category")) for r in risks if _is_present(r.get("risk_category"))})
    risk_ratings = sorted({str(r.get("risk_rating")) for r in risks if _is_present(r.get("risk_rating"))})
    scenario_links = sorted({str(r.get("scenario_analysis_link")) for r in risks if _is_present(r.get("scenario_analysis_link"))})
    mitigation_actions = sorted({str(r.get("mitigation_actions")) for r in risks if _is_present(r.get("mitigation_actions"))})

    integrated_count = sum(1 for r in risks if r.get("erm_integrated_flag") is True)
    changed_count = sum(1 for r in risks if r.get("changed_since_prior_period") is True)

    # Keep only the most useful examples for prompt compactness.
    material_risk_examples = []
    rating_priority = {"critical": 4, "high": 3, "medium": 2, "low": 1}
    sorted_risks = sorted(
        risks,
        key=lambda r: (
            rating_priority.get(str(r.get("risk_rating", "")).lower(), 0),
            float(r.get("financial_impact_meur") or 0)
        ),
        reverse=True,
    )
    for r in sorted_risks[:5]:
        material_risk_examples.append({
            "risk_id": r.get("risk_id"),
            "risk_name": r.get("risk_name"),
            "risk_category": r.get("risk_category"),
            "risk_rating": r.get("risk_rating"),
            "time_horizon": r.get("time_horizon"),
            "monitoring_frequency": r.get("monitoring_frequency"),
            "erm_integrated_flag": r.get("erm_integrated_flag"),
            "scenario_analysis_link": r.get("scenario_analysis_link"),
            "mitigation_actions": r.get("mitigation_actions"),
        })

    return {
        "risk_register_available": True,
        "reporting_year": year,
        "risk_count": len(risks),
        "erm_integrated_count": integrated_count,
        "changed_since_prior_period_count": changed_count,
        "monitoring_frequencies": frequencies,
        "risk_categories": risk_categories,
        "risk_ratings": risk_ratings,
        "scenario_analysis_links": scenario_links,
        "mitigation_actions": mitigation_actions[:8],
        "material_risk_examples": material_risk_examples,
        "process_flow_instruction": (
            "Write management responsibility as a process flow: identify climate risks in the climate risk register; "
            "classify them by category, time horizon and rating; monitor them at the recorded quarterly or semi-annual "
            "frequency; link relevant risks to scenario analysis where a scenario link exists; define mitigation actions; "
            "and use ERM integration to support management monitoring and board or committee review where required. "
            "Do not invent a formal escalation threshold unless explicitly provided."
        )
    }


def extract_governance_evidence(payload: dict) -> dict:

    gov_records = payload.get("governance", [])
    board_minutes = payload.get("board_minutes", [])
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})

    gov_by_year = {
        str(r["reporting_year"]): r
        for r in gov_records
        if isinstance(r, dict) and "reporting_year" in r
    }

    gov_trend = []
    for year in ["2022", "2023", "2024"]:
        if year in gov_by_year:
            g = gov_by_year[year]
            gov_trend.append({
                "year": int(year),
                "esg_committee_meetings": g.get("esg_committee_meetings_per_year"),
                "board_climate_expertise_pct": g.get("board_climate_expertise_pct"),
                "ceo_esg_compensation_pct": g.get("ceo_esg_compensation_pct"),
                "all_exec_climate_remuneration_pct": g.get("all_exec_climate_remuneration_pct"),
                "climate_on_board_agenda_pct": g.get("climate_on_board_agenda_pct"),
                "management_committee_name": g.get("management_committee_name"),
                "board_full_meeting_frequency": g.get("board_full_meeting_frequency"),
            })

    PRIORITY_TOPICS = [
        "scenario", "transition", "net_zero", "target", "carbon_credit",
        "remuneration", "tcfd", "green_finance", "physical_risk", "risk", "esg"
    ]

    def decision_score(m: dict) -> int:
        topics = str(m.get("climate_topics_discussed", "")).lower()
        decision = str(m.get("decision_summary", "")).lower()
        ifrs = str(m.get("ifrs_s2_para_evidence", "")).lower()
        score = sum(1 for t in PRIORITY_TOPICS if t in topics or t in decision)
        if "6(a)(v)" in ifrs:
            score += 2
        if str(m.get("committee_type", "")).lower() == "full_board":
            score += 1
        return score

    minutes_2024 = [
        m for m in board_minutes
        if isinstance(m, dict)
        and _safe_int(m.get("reporting_year")) == 2024
        and m.get("decision_made_flag") is True
        and _is_present(m.get("decision_summary"))
        and _is_present(m.get("meeting_id"))
    ]

    # Deduplicate by decision text only; keep the highest-scoring exact record and retain meeting_id.
    best_by_decision = {}
    for m in minutes_2024:
        decision_text = re.sub(r"\s+", " ", str(m.get("decision_summary", "")).lower().strip())
        current = best_by_decision.get(decision_text)
        if current is None or decision_score(m) > decision_score(current):
            best_by_decision[decision_text] = m

    selected_decisions = []
    for m in sorted(best_by_decision.values(), key=decision_score, reverse=True)[:6]:
        selected_decisions.append({
            "meeting_id": m.get("meeting_id"),
            "date": m.get("meeting_date"),
            "committee": m.get("committee_name"),
            "committee_type": m.get("committee_type"),
            "topics_discussed": m.get("climate_topics_discussed"),
            "decision": m.get("decision_summary"),
            "ifrs_evidence_para": m.get("ifrs_s2_para_evidence"),
            "internal_ref": f"[REF:{m.get('meeting_id')}]",
        })

    gov_2024 = gov_by_year.get("2024", {})

    # Evidence gap assessment for strict governance disclosure.
    # These are intentionally conservative: the writer may state a limitation, but must not invent missing details.
    governance_instrument_fields = [
        "committee_charter", "committee_terms_of_reference", "board_mandate", "esg_committee_mandate",
        "formal_climate_mandate", "governance_policy_reference", "committee_charter_climate_mandate"
    ]
    formal_mandate_available = any(_is_present(gov_2024.get(f)) for f in governance_instrument_fields)

    tradeoff_terms = ["tradeoff", "trade-off", "capital allocation", "profitability", "cost", "risk appetite", "competing"]
    tradeoff_decisions = [
        m for m in minutes_2024
        if any(term in str(m.get("decision_summary", "")).lower() or term in str(m.get("climate_topics_discussed", "")).lower() for term in tradeoff_terms)
    ]
    board_tradeoff_evidence_available = len(tradeoff_decisions) > 0

    skills_process_fields = [
        "skills_matrix", "skills_assessment_process", "board_skills_review", "skills_adequacy_assessment",
        "director_training_frequency", "training_hours", "skills_gap_analysis"
    ]
    skills_adequacy_process_available = any(_is_present(gov_2024.get(f)) for f in skills_process_fields)

    assurance_scope = str(gov_2024.get("assurance_scope", ""))
    financed_emissions_2024 = reporting_kpis.get("financed_emissions_2024_tco2e")
    assurance_scope_limitation = {
        "assurance_scope": assurance_scope,
        "external_assurance": gov_2024.get("external_assurance"),
        "provider": gov_2024.get("assurance_provider"),
        "standard": gov_2024.get("assurance_standard"),
        "financed_emissions_2024_tco2e": financed_emissions_2024,
        "financed_emissions_in_scope": "financed" in assurance_scope.lower() or "scope 3" in assurance_scope.lower(),
        "instruction": (
            "State that assurance covers only the stated scope. If the stated scope is Scope 1 and 2 emissions, "
            "do not imply financed emissions or other Scope 3 categories are assured. For a bank, explicitly clarify "
            "that financed emissions are outside the stated assurance scope based on available evidence."
        )
    }

    return {
        "bank": {
            "name": bank.get("bank_name"),
            "country": bank.get("country"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "reporting_year": 2024,
        "comparative_years": [2022, 2023],
        "governance_2024": {
            "board_size": gov_2024.get("board_size"),
            "independent_directors_pct": gov_2024.get("independent_directors_pct"),
            "esg_committee_exists": gov_2024.get("esg_committee_exists"),
            "esg_committee_meetings_per_year": gov_2024.get("esg_committee_meetings_per_year"),
            "board_climate_expertise_pct": gov_2024.get("board_climate_expertise_pct"),
            "ceo_compensation_esg_linked": gov_2024.get("ceo_compensation_esg_linked"),
            "ceo_esg_compensation_pct": gov_2024.get("ceo_esg_compensation_pct"),
            "all_exec_climate_remuneration_pct": gov_2024.get("all_exec_climate_remuneration_pct"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "climate_on_board_agenda_pct": gov_2024.get("climate_on_board_agenda_pct"),
            "board_full_meeting_frequency": gov_2024.get("board_full_meeting_frequency"),
            "management_committee_name": gov_2024.get("management_committee_name"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "skills_development_programme": gov_2024.get("skills_development_programme"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "external_assurance": gov_2024.get("external_assurance"),
            "assurance_provider": gov_2024.get("assurance_provider"),
            "assurance_scope": gov_2024.get("assurance_scope"),
            "assurance_standard": gov_2024.get("assurance_standard"),
            "tcfd_aligned": gov_2024.get("tcfd_aligned"),
            "ifrs_s2_aligned": gov_2024.get("ifrs_s2_aligned"),
        },
        "governance_trend": gov_trend,
        "management_process_evidence": extract_management_process_evidence(payload, year=2024),
        "board_decisions_2024": selected_decisions,
        "strict_governance_evidence": {
            "formal_governance_mandate_available": formal_mandate_available,
            "formal_governance_mandate_instruction": (
                "Do not claim the ESG & Sustainability Committee has a formal climate mandate unless charter/terms-of-reference evidence is provided. "
                "If no formal instrument is available, say the payload evidences committee activity and meeting frequency but does not include the committee charter or terms of reference."
            ),
            "board_tradeoff_evidence_available": board_tradeoff_evidence_available,
            "tradeoff_decisions": tradeoff_decisions[:3],
            "board_tradeoff_instruction": (
                "Discuss board trade-offs only if explicit trade-off evidence exists. If not, state that the board decision evidence identifies climate-related decisions, "
                "but does not describe specific trade-offs such as profitability, capital allocation, implementation cost, risk appetite or competing strategic priorities."
            ),
            "skills_adequacy_process_available": skills_adequacy_process_available,
            "skills_adequacy_instruction": (
                "Use the board climate expertise percentage and skills development programme as outcome/activity evidence. "
                "Do not invent a formal skills adequacy assessment process. If no skills assessment evidence exists, say the payload does not describe a formal board skills adequacy assessment process."
            ),
            "assurance_scope_limitation": assurance_scope_limitation,
        },
        "interpretation_notes": {
            "climate_on_board_agenda_pct": (
                "This figure represents the percentage of board meetings during the year "
                "where climate-related topics appeared on the agenda. It does NOT mean "
                "percentage of agenda time devoted to climate."
            ),
            "management_committee_names": (
                "Committee names are recorded by year only. The evidence does not prove that "
                "one committee evolved into, replaced, or was renamed as another. State the 2024 "
                "committee name and, if comparative names are used, present them neutrally."
            ),
            "board_decision_traceability": (
                "Each selected decision includes a meeting_id for audit traceability. Meeting IDs "
                "may be used internally but should not be printed in the final report unless required."
            ),
            "avoid_duplication": (
                "Do not list the same board decisions twice. Board oversight should summarise decision governance; "
                "the detailed dated list belongs only in the Board and committee decisions subsection."
            )
        }
    }


evidence = extract_governance_evidence(payload)

print(f"Evidence extracted for: {evidence['bank']['name']}")
print(f"Board decisions selected: {len(evidence['board_decisions_2024'])}")
print(f"Trend years: {[t['year'] for t in evidence['governance_trend']]}")
print(f"Risk-register records for management process: {evidence['management_process_evidence'].get('risk_count')}")
print("Strict governance evidence flags:")
for k, v in evidence["strict_governance_evidence"].items():
    if isinstance(v, bool):
        print(f"- {k}: {v}")
print("Selected decisions with internal refs:")
for d in evidence["board_decisions_2024"]:
    print(f"- {d['date']} | {d['committee']} | {d['decision']} | {d['internal_ref']}")


Evidence extracted for: Eurolux Universal Bank AG
Board decisions selected: 5
Trend years: [2022, 2023, 2024]
Risk-register records for management process: 8
Strict governance evidence flags:
- formal_governance_mandate_available: False
- board_tradeoff_evidence_available: False
- skills_adequacy_process_available: False
Selected decisions with internal refs:
- 2024-01-24 | Full Board | Approved 2024 ESG report for publication | [REF:MTG-BANK01-FB-2024-001]
- 2024-04-15 | ESG & Sustainability Committee | Approved climate scenario analysis methodology | [REF:MTG-BANK01-ESG-2024-011]
- 2024-04-15 | ESG & Sustainability Committee | Approved carbon credit procurement budget | [REF:MTG-BANK01-ESG-2024-010]
- 2024-10-07 | Full Board | Endorsed updated transition plan | [REF:MTG-BANK01-FB-2024-004]
- 2024-11-01 | Full Board | Endorsed net-zero interim target revision | [REF:MTG-BANK01-FB-2024-007]


In [4]:
# ── STATE DEFINITION ─────────────────────────────────────────
class GovernanceState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

In [5]:
# ── GOVERNANCE REQUIREMENTS ─────────────────────────────────
# IFRS references are used internally for coverage only.
# The final markdown headings and body must NOT include IFRS paragraph references.

IFRS_GOVERNANCE_REQUIREMENTS = """
STRICT GOVERNANCE DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Board oversight:
  - Describe board oversight of climate-related risks and opportunities.
  - Distinguish activity evidence from formal mandate evidence.
  - If committee charter / terms of reference / formal mandate evidence is not provided, state that the available evidence shows committee activity and meeting frequency, but does not include the formal governance instrument.
  - Explain how the board is informed about climate matters, using climate reporting cadence and board agenda evidence.
  - Explain how climate is considered in oversight of strategy, major transactions, risk management, metrics and targets.
  - Discuss board-level trade-offs only if evidenced. If no trade-off evidence exists, state that board decisions are evidenced but specific trade-offs are not described in the available minutes evidence.
  - Do not list the detailed board decisions here; summarise and point to the dedicated decisions subsection.

Management responsibility:
  - Which management body or role is responsible for climate-related risks and opportunities.
  - Write a process flow, not only an inventory: risk identification, register recording, classification, monitoring frequency, scenario links, mitigation actions, and ERM integration.
  - Be careful with escalation wording: only state a formal escalation threshold if explicit evidence exists. Otherwise use safe wording about board or committee review where required.

Climate skills and competencies:
  - Use board climate expertise and skills development programme evidence.
  - Describe the skills adequacy assessment process only if evidence exists.
  - If no skills adequacy process evidence exists, state that the payload evidences expertise percentage and skills development activity, but does not describe a formal skills adequacy assessment process.

Remuneration:
  - Explain whether and how climate-related performance metrics are incorporated into remuneration.
  - Cite percentage of CEO and all-executive remuneration linked to ESG/climate metrics where available.

Board and committee decisions:
  - Include the detailed dated list only in this subsection.
  - Use exact date, committee and decision pairings from board_decisions_2024.
  - Do not repeat the same detailed decisions in Board oversight.

External assurance and controls:
  - State assurance provider, standard, level and exact scope.
  - Explain limited assurance safely as lower assurance than reasonable assurance.
  - Do not imply assurance covers metrics outside the stated scope.
  - For a bank, if assurance covers only Scope 1 and Scope 2 emissions, clarify that financed emissions / Scope 3 are outside the stated assurance scope based on available evidence.
"""


In [6]:
# ── WRITER SYSTEM PROMPT ─────────────────────────────────────
WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Governance section
of an IFRS S1/S2 aligned climate disclosure report for a commercial bank.

WRITING STANDARDS:
- Formal, third-person professional disclosure language suitable for publication.
- Specific and data-driven — cite exact figures, dates, and percentages.
- Every quantitative claim must come from the provided evidence — never invent numbers.
- Use IFRS requirements internally for coverage, but DO NOT put IFRS paragraph references in subsection headings or body text.
- No vague language such as "demonstrates commitment" unless backed by concrete evidence.
- Avoid overly strong assurance/control language such as "ensuring"; prefer "supporting", "providing", or "helping".
- Avoid compliance conclusions such as "aligned with IFRS S2 requirements", "fully aligned", "compliant", or "ensures reliability".
- Do not add a generic limitation disclaimer at the end.
- Specific evidence boundaries are allowed and required when evidence is missing, for example: "the available evidence does not describe...".
- Do not hedge when data is clearly available.

STRICT IFRS GOVERNANCE CONTROL:
- Distinguish board-level requirements from management-level requirements.
- Do not use management process evidence to satisfy board trade-off or board mandate disclosure.
- Board oversight must address mandate/activity, information flow, strategy/major transaction oversight, trade-offs or trade-off evidence limitation, and target/decision monitoring.
- Management responsibility must be written as a process flow, not as a raw inventory of risk register facts.
- Climate skills must separate outcome metrics from the process for assessing skills adequacy. Do not invent a skills adequacy process.
- External assurance must state the exact assurance scope and clarify when financed emissions / Scope 3 are outside that scope.
- Do not duplicate the detailed board decision list across multiple subsections.

HALLUCINATION CONTROL:
- Do not infer that a committee evolved, was renamed, replaced, strengthened, or specialized across years unless the evidence explicitly says so.
- If committee names differ by year, state only that different names are recorded across the comparative period.
- Do not create a transformation narrative from time-series values.
- Use board decision dates only from board_decisions_2024 and keep the date-decision-committee pairing exactly as provided.
- For escalation, do not write "significant or material climate risks are escalated" unless the evidence explicitly provides escalation criteria or route.
  Safer wording: "The risk register and ERM integration support management monitoring and board or committee review where required."
- For remuneration, avoid interpretive wording such as "demonstrates progressive integration". Use factual wording such as "indicates increased use".
- For assurance, explain limited assurance as lower than reasonable assurance. Do not call it "moderate assurance".

OUTPUT FORMAT:
Return markdown with exactly this subsection structure and NO IFRS paragraph references in headings or body text:

### Governance

#### Board oversight
...content...

#### Management responsibility
...content...

#### Climate skills and competencies
...content...

#### Remuneration and climate incentives
...content...

#### Board and committee decisions during 2024
...content...

#### External assurance and controls
...content...
""".strip()


def build_writer_prompt(evidence: dict, judge_feedback: str = None) -> str:
    is_revision = judge_feedback is not None

    base_instructions = f"""
BANK: {evidence['bank']['name']} ({evidence['bank']['country']})
REPORTING YEAR: {evidence['reporting_year']}
COMPARATIVE YEARS: {evidence['comparative_years']}

GOVERNANCE REQUIREMENTS FOR THIS SECTION:
{IFRS_GOVERNANCE_REQUIREMENTS}

EVIDENCE (use ONLY this data):
{json.dumps(evidence, indent=2, ensure_ascii=False)}

CRITICAL INTERPRETATION RULES:
1. climate_on_board_agenda_pct = percentage of board MEETINGS where climate was on the agenda.
   NOT percentage of agenda time. Write it as: "climate featured on the agenda of X% of board meetings".
2. Committee names are evidence values by year only. Do NOT say "evolved from", "renamed from",
   "replaced", "progressively strengthened", or "specialized" unless explicit evidence says so.
   For BANK01, write: "In 2024, management-level climate governance is led by the Climate Risk Management Committee."
   If mentioning prior years, say only: "The recorded management committee name was X in 2022 and Y in 2023."
3. For board oversight, include a formal mandate sentence only if strict_governance_evidence.formal_governance_mandate_available is true.
   If false, state: "The available payload evidences ESG & Sustainability Committee activity and meeting frequency, but does not include the committee charter or terms of reference."
4. For board trade-offs, do not invent trade-offs. If strict_governance_evidence.board_tradeoff_evidence_available is false, state:
   "The available board minutes evidence identifies climate-related decisions but does not describe specific trade-offs considered by the board, such as profitability, capital allocation, implementation cost or risk appetite impacts."
5. For management responsibility, write a process flow using management_process_evidence:
   identify risks in the climate risk register; classify by category/time horizon/rating; monitor quarterly or semi-annually;
   link selected risks to scenario analysis; define mitigation actions; use ERM integration for management monitoring and board/committee review where required.
6. For climate skills, cite the board climate expertise trend and skills development programme.
   If strict_governance_evidence.skills_adequacy_process_available is false, state that the payload does not describe a formal board skills adequacy assessment process.
7. Include year-on-year trends for: board climate expertise %, CEO ESG compensation %, ESG committee meeting frequency, climate on board agenda %.
8. Board decisions: in Board oversight, only summarise that climate-related decisions are detailed below. Put the full dated list ONLY in "Board and committee decisions during 2024".
9. Cite at least 4 specific board/committee decisions from board_decisions_2024 with dates in the dedicated decision subsection.
   Use the exact date, committee, and decision pairing provided. Meeting IDs are internal traceability refs; do not print them.
10. For remuneration: cite both CEO ESG compensation % AND all-executive climate remuneration %.
    Prefer "indicates increased use" over "demonstrates progressive integration".
11. Explain limited assurance safely: limited assurance provides a lower level of assurance than reasonable assurance,
    based on procedures performed over the stated assurance scope. Do NOT say "moderate assurance".
12. Assurance scope: if stated scope is Scope 1 and 2 emissions, explicitly say financed emissions and other Scope 3 categories are outside the stated assurance scope based on available evidence.
13. Do NOT write "the bank's disclosures are aligned with IFRS S2 requirements". Safer wording:
    "The governance evidence is presented with reference to TCFD recommendations and IFRS governance disclosure requirements."
14. The final markdown must not contain visible IFRS paragraph references such as [IFRS S2 §6(a)], [IFRS S2 §7], §6(a), §6(b), or §7 in headings or body text.
"""

    if is_revision:
        return f"""
{base_instructions}

JUDGE FEEDBACK TO ADDRESS IN THIS REVISION:
{judge_feedback}

REVISION RULES:
- Fix every issue the judge flagged.
- Do not remove content that was not criticised.
- Do not add information not present in the evidence.
- Preserve the required subsection structure with NO visible IFRS paragraph references.

Write the revised governance section now.
""".strip()

    return f"""
{base_instructions}

Write the complete governance section now.
Follow the exact subsection structure specified in your instructions.
""".strip()


In [7]:
# ── JUDGE SYSTEM PROMPT ──────────────────────────────────────
JUDGE_SYSTEM = """
You are a strict IFRS S1/S2 compliance reviewer and ESG audit specialist.
Your job is to identify genuine gaps in a governance disclosure section — not to reward fluent writing.

CORE PRINCIPLE:
- A limitation statement reduces hallucination risk, but it is NOT equivalent to direct evidence.
- Do not treat "limitation disclosed" as full coverage.
- A section can be approved with limitations, but it should not receive a 9 or 10 when material IFRS governance elements are covered only by limitation statements.

SCORING ANCHOR:
  10: Audit-ready. All required governance elements are directly evidenced, specific, complete, non-duplicative, and no limitation statement is needed.
  9: Strong. Minor wording issues only. No material missing evidence and no material requirement covered only by limitation.
  8: Good but incomplete. One material governance element is addressed through a limitation statement rather than direct evidence.
  7: Usable draft with limitations. Two or more material governance elements are addressed through limitation statements, or one governance process is thin but honestly disclosed.
  6: Needs revision. Missing or weak coverage of one core IFRS governance requirement.
  5 or below: Not approved. Unsupported claims, wrong numbers, major IFRS coverage failure, or hallucination risk.

MATERIAL LIMITATION ITEMS:
These should be tracked separately as evidence vs limitation:
- formal governance mandate / committee charter or terms of reference
- board consideration of climate-related trade-offs
- formal board skills adequacy assessment process
- formal escalation thresholds for climate risks
- assurance scope excludes financed emissions / Scope 3 where those are material for a bank

A score of 9 or 10 requires ALL of the following to be true:
- All 6 subsections present with substantive content.
- No visible IFRS paragraph references appear in subsection headings or body text.
- climate_on_board_agenda_pct correctly interpreted as meeting frequency, not agenda time.
- At least 4 distinct board/committee decisions cited with exact dates in the dedicated decisions subsection.
- Detailed board decisions are not duplicated in the Board oversight subsection.
- Year-on-year trends present for board expertise, CEO compensation, ESG committee meetings and climate agenda frequency.
- Both CEO ESG % and all-executive climate % cited in remuneration.
- Formal governance mandate is directly evidenced, not merely disclosed as unavailable.
- Board trade-offs are directly evidenced, not merely disclosed as unavailable.
- Management responsibility is written as a process flow: risk identification/register, classification, monitoring frequency, scenario links, mitigation actions and ERM integration.
- Escalation thresholds or route are directly evidenced if claimed; otherwise a limitation is stated.
- No unsupported committee evolution / renaming / strengthening narrative.
- Skills adequacy process is directly evidenced, not merely disclosed as unavailable.
- Limited assurance is explained safely as lower assurance than reasonable assurance.
- Assurance scope limitation is clearly disclosed, especially that financed emissions / Scope 3 are outside the stated assurance scope when applicable.
- No generic limitation disclaimer at the end.
- No unsupported claims such as "ensures", "guarantees", "fully aligned", "fully resilient", "compliant", or "aligned with IFRS S2 requirements".

SCORE CAPS:
- If any required subsection is missing: maximum score 6.
- If any unsupported strong claim or hallucination appears: maximum score 6.
- If detailed decisions are duplicated: maximum score 7.
- If management is only an inventory and not a process flow: maximum score 7.
- If one material limitation item is present: maximum score 8.
- If two or three material limitation items are present: maximum score 8.
- If four or more material limitation items are present: maximum score 7.
- If the output is evidence-based but has multiple disclosed limitations, it should usually be "approved_with_limitations", not fully approved.

You must return valid JSON only — no other text.
""".strip()


def build_judge_prompt(draft: str, evidence: dict) -> str:

    gov_2024 = evidence.get("governance_2024", {})
    trend = evidence.get("governance_trend", [])
    decisions = evidence.get("board_decisions_2024", [])
    management_process = evidence.get("management_process_evidence", {})
    strict = evidence.get("strict_governance_evidence", {})

    return f"""
Evaluate this governance section draft against the strict governance requirements.

DRAFT TO EVALUATE:
{draft}

KEY DATA AVAILABLE TO THE WRITER:
- Board size: {gov_2024.get('board_size')} members
- Independent directors: {gov_2024.get('independent_directors_pct')}%
- ESG committee meetings 2024: {gov_2024.get('esg_committee_meetings_per_year')}
- Board climate expertise: 2022={next((t['board_climate_expertise_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['board_climate_expertise_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('board_climate_expertise_pct')}%
- CEO ESG compensation: 2022={next((t['ceo_esg_compensation_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['ceo_esg_compensation_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('ceo_esg_compensation_pct')}%
- All-exec climate remuneration 2024: {gov_2024.get('all_exec_climate_remuneration_pct')}%
- Climate on board agenda: 2022={next((t['climate_on_board_agenda_pct'] for t in trend if t['year']==2022), 'N/A')}%, 2023={next((t['climate_on_board_agenda_pct'] for t in trend if t['year']==2023), 'N/A')}%, 2024={gov_2024.get('climate_on_board_agenda_pct')}%
- Management committee 2024: {gov_2024.get('management_committee_name')}
- Management process evidence: {json.dumps(management_process, ensure_ascii=False)}
- Strict governance evidence flags: {json.dumps(strict, ensure_ascii=False)}
- External assurance: {gov_2024.get('external_assurance')} by {gov_2024.get('assurance_provider')} under {gov_2024.get('assurance_standard')}; scope={gov_2024.get('assurance_scope')}
- Available board decisions with meeting IDs: {json.dumps(decisions, ensure_ascii=False)}

VERIFICATION CHECKLIST — answer each with true/false. Separate direct evidence from limitation statements:
1. all_six_subsections_present: Are all 6 required subsections present?
2. no_visible_ifrs_refs: Are there NO visible IFRS paragraph references in headings or body text?
3. agenda_pct_correct: Is climate_on_board_agenda_pct described as meeting frequency, not agenda time?
4. four_distinct_decisions: Are at least 4 distinct board/committee decisions cited with exact dates in the dedicated decisions subsection?
5. no_duplicate_decisions: Are detailed dated decisions absent from Board oversight and listed only in the dedicated decisions subsection?
6. yoy_trends_present: Are year-on-year trends present for expertise, CEO compensation, ESG committee meetings, and climate agenda frequency?
7. both_remuneration_figures: Are both CEO ESG % and all-exec climate % cited?
8a. formal_mandate_evidenced: Does the text describe a formal governance mandate, charter, terms of reference, or other formal governance instrument from direct evidence?
8b. formal_mandate_limitation_used: If no formal mandate evidence exists, does the text state that committee activity is evidenced but the formal charter/terms of reference is not documented?
9a. board_tradeoffs_evidenced: Does the text describe specific board-level trade-offs from the minutes/evidence?
9b. board_tradeoffs_limitation_used: If no trade-off evidence exists, does the text state that specific trade-offs are not described in the available minutes/evidence?
10. management_process_flow: Does management responsibility read as a process flow, not merely an inventory of risks?
11a. escalation_thresholds_evidenced: Does the text disclose formal escalation thresholds/routes only if explicitly evidenced?
11b. escalation_threshold_limitation_used: If thresholds/routes are not evidenced, does the text state that formal escalation thresholds are not specified?
12. escalation_not_overclaimed: Does the section avoid formal escalation claims unless explicit evidence supports them?
13. no_unsupported_committee_evolution: No claim that committees evolved/renamed/replaced/strengthened unless explicitly evidenced?
14a. skills_adequacy_process_evidenced: Does the text disclose a formal skills adequacy assessment process from direct evidence?
14b. skills_adequacy_limitation_used: If no skills adequacy process evidence exists, does the text state that only expertise percentage and skills development activity are evidenced?
15. assurance_explained_safely: Limited assurance explained without "moderate assurance" or overstated assurance conclusions?
16. assurance_scope_limitation: Does the assurance section state the exact scope and avoid implying financed emissions/Scope 3 are assured when the scope is only Scope 1 and 2?
17. financed_emissions_outside_assurance_scope: If financed emissions are material and not in the assurance scope, is this explicitly stated?
18. no_limitation_disclaimer: No generic limitation disclaimer at the end? Specific evidence-boundary statements are allowed.
19. no_unsupported_strong_claims: No unsupported words/phrases like ensures, guarantees, fully aligned, compliant, aligned with IFRS S2 requirements?

COUNT false checklist items.
COUNT material limitation items used among:
- formal_mandate_limitation_used
- board_tradeoffs_limitation_used
- escalation_threshold_limitation_used
- skills_adequacy_limitation_used
- financed_emissions_outside_assurance_scope

Important scoring rule:
- A limitation item can be true and acceptable, but it still lowers completeness.
- Do NOT give 9 or 10 when material_limitation_count > 0.
- If material_limitation_count >= 4, maximum score is 7.
- If material_limitation_count is 1-3, maximum score is 8.

Return this exact JSON structure:
{{
  "overall_score": <integer 1-10 after applying score caps>,
  "raw_score_before_caps": <integer 1-10 before applying limitation caps>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approval_status is approved or approved_with_limitations, else false>,
  "checklist": {{
    "all_six_subsections_present": <true/false>,
    "no_visible_ifrs_refs": <true/false>,
    "agenda_pct_correct": <true/false>,
    "four_distinct_decisions": <true/false>,
    "no_duplicate_decisions": <true/false>,
    "yoy_trends_present": <true/false>,
    "both_remuneration_figures": <true/false>,
    "formal_mandate_evidenced": <true/false>,
    "formal_mandate_limitation_used": <true/false>,
    "board_tradeoffs_evidenced": <true/false>,
    "board_tradeoffs_limitation_used": <true/false>,
    "management_process_flow": <true/false>,
    "escalation_thresholds_evidenced": <true/false>,
    "escalation_threshold_limitation_used": <true/false>,
    "escalation_not_overclaimed": <true/false>,
    "no_unsupported_committee_evolution": <true/false>,
    "skills_adequacy_process_evidenced": <true/false>,
    "skills_adequacy_limitation_used": <true/false>,
    "assurance_explained_safely": <true/false>,
    "assurance_scope_limitation": <true/false>,
    "financed_emissions_outside_assurance_scope": <true/false>,
    "no_limitation_disclaimer": <true/false>,
    "no_unsupported_strong_claims": <true/false>,
    "false_count": <integer>,
    "material_limitation_count": <integer>
  }},
  "main_issues": [<list of specific issues found>],
  "required_fixes": [<specific actionable instructions for the reviser>]
}}
""".strip()


In [8]:
# ── DETERMINISTIC RULE CHECKS ────────────────────────────────
# These run before the LLM judge and can override it.
# They catch known failure modes from the governance evaluation.


def _section_text(draft: str, heading: str, next_headings: list[str]) -> str:
    """Extract text under a markdown h4 heading."""
    lower = draft.lower()
    h = heading.lower()
    start = lower.find(h)
    if start == -1:
        return ""
    start = start + len(h)
    end_candidates = []
    for nh in next_headings:
        pos = lower.find(nh.lower(), start)
        if pos != -1:
            end_candidates.append(pos)
    end = min(end_candidates) if end_candidates else len(draft)
    return draft[start:end]


def rule_check(draft: str) -> dict:
    text = draft.lower()

    required_subsections = [
        "#### board oversight",
        "#### management responsibility",
        "#### climate skills and competencies",
        "#### remuneration and climate incentives",
        "#### board and committee decisions",
        "#### external assurance",
    ]

    missing_subsections = [s for s in required_subsections if s not in text]

    unsupported_evolution_patterns = [
        "evolved from",
        "evolved into",
        "progressive strengthening",
        "progressively strengthening",
        "progressive integration",
        "demonstrates a progressive integration",
        "specialization of the bank",
        "specialisation of the bank",
        "was renamed",
        "renamed as",
        "replaced by",
        "transformed into",
    ]

    visible_ifrs_patterns = [
        "[ifrs",
        "ifrs s2 §",
        "ifrs s1 §",
        "§6(a)",
        "§6(b)",
        "§6(a)(v)",
        "§7",
        "§8",
        "§9",
    ]

    # Duplication: detailed decisions should appear only in the dedicated decision subsection.
    board_oversight = _section_text(
        draft,
        "#### Board oversight",
        [
            "#### Management responsibility",
            "#### Climate skills and competencies",
            "#### Remuneration and climate incentives",
            "#### Board and committee decisions during 2024",
            "#### External assurance and controls",
        ]
    ).lower()
    decision_like_dates_in_board_oversight = len(re.findall(
        r"\b\d{1,2}\s+(january|february|march|april|may|june|july|august|september|october|november|december)\b",
        board_oversight
    ))
    duplicated_decisions_in_board_oversight = decision_like_dates_in_board_oversight >= 2

    # Specific strict evidence-boundary checks. These do not prove quality, but catch obvious missing language.
    formal_mandate_language_present = any(
        phrase in text for phrase in [
            "charter", "terms of reference", "formal mandate", "formal governance instrument",
            "does not include the committee charter", "does not include the committee's charter"
        ]
    )
    tradeoff_language_present = any(
        phrase in text for phrase in [
            "trade-off", "tradeoff", "trade-offs", "specific trade-offs", "does not describe specific trade-offs",
            "capital allocation", "risk appetite", "implementation cost", "profitability"
        ]
    )
    skills_process_language_present = any(
        phrase in text for phrase in [
            "skills adequacy", "skills assessment", "skills matrix", "formal board skills", "does not describe a formal board skills"
        ]
    )
    assurance_scope_limitation_present = (
        "scope 1" in text and "scope 2" in text and
        ("financed emissions" in text or "scope 3" in text) and
        any(p in text for p in ["outside the stated assurance scope", "not within the stated assurance scope", "does not extend"])
    )

    hard_fails = {
        "visible_ifrs_references": any(p in text for p in visible_ifrs_patterns),
        "agenda_time_misinterpretation": (
            "agenda time" in text or
            "% of the board's agenda" in text or
            "dedicated to climate" in text
        ),
        "generic_limitation_disclaimer": any(
            phrase in text for phrase in [
                "we acknowledge this limitation",
                "absence of prepared evidence",
                "unable to provide",
                "will strive to provide",
                "this section acknowledges",
            ]
        ),
        "unsupported_committee_evolution": any(p in text for p in unsupported_evolution_patterns),
        "overclaimed_escalation": "significant or material climate risks are escalated" in text,
        "unsafe_assurance_language": any(
            phrase in text for phrase in [
                "moderate level of assurance",
                "moderate assurance",
                "ensuring the reliability",
                "ensures the reliability",
            ]
        ),
        "unsupported_alignment_or_compliance_claim": any(
            phrase in text for phrase in [
                "disclosures are aligned with tcfd",
                "disclosures are aligned with ifrs",
                "aligned with ifrs s2 requirements",
                "fully aligned",
                "compliant with",
                "complies with",
            ]
        ),
        "overstrong_control_language": any(
            phrase in text for phrase in [
                "ensuring regular updates",
                "ensuring transparency",
                "guarantees",
                "fully resilient",
            ]
        ),
        "duplicate_decision_listing": duplicated_decisions_in_board_oversight,
        "missing_formal_mandate_boundary": not formal_mandate_language_present,
        "missing_board_tradeoff_boundary": not tradeoff_language_present,
        "missing_skills_adequacy_boundary": not skills_process_language_present,
        "missing_assurance_scope_limitation": not assurance_scope_limitation_present,
    }

    structure_ok = len(missing_subsections) == 0
    content_ok = not any(hard_fails.values())

    return {
        "passed": structure_ok and content_ok,
        "structure_ok": structure_ok,
        "content_ok": content_ok,
        "missing_subsections": missing_subsections,
        "hard_fails": {k: v for k, v in hard_fails.items() if v},
        "required_fixes": (
            [f"Add missing subsection: {s}" for s in missing_subsections] +
            [f"Fix hard fail: {k}" for k, v in hard_fails.items() if v]
        )
    }


In [9]:
# ── LANGGRAPH NODES ──────────────────────────────────────────

def writer_node(state: GovernanceState) -> GovernanceState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_writer_prompt(state["evidence"], judge_feedback=feedback)

    draft = call_llm(
        system_prompt=WRITER_SYSTEM,
        user_prompt=prompt,
        temperature=0.2
    )

    print(f"\n{'='*50}")
    print(f"WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {
        **state,
        "draft": draft.strip(),
        "status": "judging"
    }



# ── STRICT SCORE CAPS ────────────────────────────────────────
def strict_governance_score(raw_score: int, checklist: dict) -> tuple[int, str, int]:
    """
    Convert a potentially generous LLM score into a stricter audit-style score.
    Limitation statements are good for honesty, but they reduce completeness.
    Returns: (final_score, cap_reason, material_limitation_count)
    """
    score = int(raw_score or 0)

    limitation_keys = [
        "formal_mandate_limitation_used",
        "board_tradeoffs_limitation_used",
        "escalation_threshold_limitation_used",
        "skills_adequacy_limitation_used",
        "financed_emissions_outside_assurance_scope",
    ]
    material_limitation_count = sum(bool(checklist.get(k, False)) for k in limitation_keys)

    cap_reasons = []

    # Missing direct evidence but disclosed as limitation still caps score.
    if material_limitation_count >= 4:
        if score > 7:
            cap_reasons.append(f"Capped at 7 because {material_limitation_count} material requirements are addressed through limitations rather than direct evidence")
        score = min(score, 7)
    elif material_limitation_count >= 1:
        if score > 8:
            cap_reasons.append(f"Capped at 8 because {material_limitation_count} material requirement(s) are addressed through limitations rather than direct evidence")
        score = min(score, 8)

    # Hard caps for actual quality failures.
    hard_failure_caps = [
        (not checklist.get("all_six_subsections_present", True), 6, "required subsection missing"),
        (not checklist.get("no_unsupported_strong_claims", True), 6, "unsupported strong claim present"),
        (not checklist.get("no_unsupported_committee_evolution", True), 6, "unsupported committee evolution/renaming narrative present"),
        (not checklist.get("no_duplicate_decisions", True), 7, "board decisions duplicated across subsections"),
        (not checklist.get("management_process_flow", True), 7, "management section is not process-based"),
        (not checklist.get("assurance_explained_safely", True), 7, "limited assurance wording is unsafe or overstated"),
        (not checklist.get("no_visible_ifrs_refs", True), 7, "visible IFRS paragraph references appear in final text"),
        (not checklist.get("assurance_scope_limitation", True), 7, "assurance scope limitation missing"),
    ]

    for condition, cap, reason in hard_failure_caps:
        if condition:
            if score > cap:
                cap_reasons.append(f"Capped at {cap}: {reason}")
            score = min(score, cap)

    return score, "; ".join(cap_reasons) if cap_reasons else "No cap applied", material_limitation_count


def derive_approval_status(score: int, checklist: dict, false_count: int, material_limitation_count: int) -> str:
    """More nuanced than approved/rejected."""
    hard_fail = (
        false_count > 0 and not (
            # limitation flags are not hard failures if handled honestly
            checklist.get("formal_mandate_limitation_used", False) or
            checklist.get("board_tradeoffs_limitation_used", False) or
            checklist.get("skills_adequacy_limitation_used", False) or
            checklist.get("escalation_threshold_limitation_used", False) or
            checklist.get("financed_emissions_outside_assurance_scope", False)
        )
    )

    if score >= 9 and material_limitation_count == 0 and false_count == 0:
        return "approved"
    if score >= 7 and not hard_fail:
        return "approved_with_limitations"
    if score >= 5:
        return "revision_required"
    return "rejected"


def judge_node(state: GovernanceState) -> GovernanceState:
    draft = state["draft"]

    # Step 1: deterministic rule checks
    rules = rule_check(draft)
    print(f"\nRule check: {'PASSED' if rules['passed'] else 'FAILED'}")
    if not rules["passed"]:
        print(f"  Issues: {rules['required_fixes']}")

    if not rules["passed"]:
        # Force a structured judge result from rule failures
        judge_result = {
            "overall_score": 4 if rules["structure_ok"] else 3,
            "evidence_support_score": 5,
            "ifrs_alignment_score": 4,
            "specificity_score": 5,
            "hallucination_risk": "medium",
            "approved": False,
            "checklist": {
                "all_six_subsections_present": rules["structure_ok"],
                "false_count": len(rules["required_fixes"])
            },
            "main_issues": rules["required_fixes"],
            "required_fixes": rules["required_fixes"],
            "rule_check_override": True
        }
        return {
            **state,
            "judge_result": judge_result,
            "status": "judging"
        }

    # Step 2: LLM judge
    judge_prompt = build_judge_prompt(draft, state["evidence"])
    judge_result = call_llm_json(
        system_prompt=JUDGE_SYSTEM,
        user_prompt=judge_prompt
    )

    # Step 3: strict score caps and approval status
    checklist = judge_result.get("checklist", {})
    false_count = int(checklist.get("false_count", 0) or 0)
    raw_score = int(judge_result.get("overall_score", 0) or 0)

    final_score, cap_reason, material_limitation_count = strict_governance_score(
        raw_score=raw_score,
        checklist=checklist
    )

    judge_result["raw_score_before_caps"] = judge_result.get("raw_score_before_caps", raw_score)
    judge_result["overall_score"] = final_score
    judge_result["score_cap_reason"] = cap_reason
    judge_result.setdefault("checklist", {})["material_limitation_count"] = material_limitation_count

    # Step 4: nuanced approval status
    approval_status = derive_approval_status(
        score=final_score,
        checklist=judge_result.get("checklist", {}),
        false_count=false_count,
        material_limitation_count=material_limitation_count,
    )
    judge_result["approval_status"] = approval_status
    judge_result["approved"] = approval_status in {"approved", "approved_with_limitations"}

    print(f"\nJudge result:")
    print(f"  Raw score: {judge_result.get('raw_score_before_caps', judge_result.get('overall_score'))}/10")
    print(f"  Final score: {judge_result.get('overall_score')}/10")
    print(f"  Approval status: {judge_result.get('approval_status')}")
    print(f"  Approved: {judge_result.get('approved')}")
    print(f"  False checks: {false_count}")
    print(f"  Material limitations: {judge_result.get('checklist', {}).get('material_limitation_count')}")
    print(f"  Cap reason: {judge_result.get('score_cap_reason')}")
    if judge_result.get("main_issues"):
        print(f"  Issues: {judge_result['main_issues']}")

    return {
        **state,
        "judge_result": judge_result,
        "status": "judging"
    }


def reviser_node(state: GovernanceState) -> GovernanceState:
    return {
        **state,
        "revision_count": state["revision_count"] + 1,
        "status": "drafting"
    }


def finalize_node(state: GovernanceState) -> GovernanceState:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)

    print(f"\n{'='*50}")
    print(f"FINALIZED")
    print(f"  Status: {'APPROVED' if approved else 'MAX REVISIONS REACHED'}")
    print(f"  Final score: {judge.get('overall_score')}/10")
    print(f"  Revisions: {state['revision_count']}")
    print(f"{'='*50}")

    return {
        **state,
        "final_section": state["draft"],
        "status": "approved" if approved else "failed"
    }


# ── ROUTING ──────────────────────────────────────────────────
def route_after_judge(state: GovernanceState) -> str:
    judge = state.get("judge_result", {})
    approved = judge.get("approved", False)
    revision_count = state.get("revision_count", 0)
    max_revisions = state.get("max_revisions", 2)

    if approved:
        return "finalize"
    if revision_count >= max_revisions:
        return "finalize"
    return "revise"

In [10]:
# ── BUILD AND COMPILE GRAPH ───────────────────────────────────
builder = StateGraph(GovernanceState)

builder.add_node("writer",   writer_node)
builder.add_node("judge",    judge_node)
builder.add_node("reviser",  reviser_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START,      "writer")
builder.add_edge("writer",   "judge")
builder.add_edge("reviser",  "writer")
builder.add_edge("finalize", END)

builder.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "revise":   "reviser",
        "finalize": "finalize",
    }
)

graph = builder.compile()
print("Graph compiled")

Graph compiled


In [11]:
# ── RUN ──────────────────────────────────────────────────────
initial_state: GovernanceState = {
    "bank_name":      bank_name,
    "evidence":       evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  2,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {}
}

print(f"Starting governance generation for: {bank_name}\n")
result = graph.invoke(initial_state)

Starting governance generation for: Eurolux Universal Bank AG


WRITER (initial draft)
Draft length: 671 words

Rule check: FAILED
  Issues: ['Fix hard fail: missing_skills_adequacy_boundary']

WRITER (revision 1)
Draft length: 689 words

Rule check: FAILED
  Issues: ['Fix hard fail: missing_skills_adequacy_boundary', 'Fix hard fail: missing_assurance_scope_limitation']

WRITER (revision 2)
Draft length: 639 words

Rule check: FAILED
  Issues: ['Fix hard fail: missing_skills_adequacy_boundary']

FINALIZED
  Status: MAX REVISIONS REACHED
  Final score: 4/10
  Revisions: 2


In [12]:
# ── OUTPUT ───────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL JUDGE RESULT")
print("="*60)
print(json.dumps(result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("GOVERNANCE SECTION")
print("="*60)
print(result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "governance_BANK01.md", "w", encoding="utf-8") as f:
    f.write(result["final_section"])

with open(output_dir / "governance_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "governance",
        "status":          result["status"],
        "approval_status": result["judge_result"].get("approval_status"),
        "raw_score":       result["judge_result"].get("raw_score_before_caps"),
        "final_score":     result["judge_result"].get("overall_score"),
        "score_cap_reason": result["judge_result"].get("score_cap_reason"),
        "revisions":       result["revision_count"],
        "approved":        result["judge_result"].get("approved"),
        "checklist":       result["judge_result"].get("checklist"),
        "issues":         result["judge_result"].get("main_issues"),
    }, f, indent=2, ensure_ascii=False)

print(f"\nSaved to outputs/governance_BANK01.md")


FINAL JUDGE RESULT
{
  "overall_score": 4,
  "evidence_support_score": 5,
  "ifrs_alignment_score": 4,
  "specificity_score": 5,
  "hallucination_risk": "medium",
  "approved": false,
  "checklist": {
    "all_six_subsections_present": true,
    "false_count": 1
  },
  "main_issues": [
    "Fix hard fail: missing_skills_adequacy_boundary"
  ],
  "required_fixes": [
    "Fix hard fail: missing_skills_adequacy_boundary"
  ],
  "rule_check_override": true
}

GOVERNANCE SECTION
### Governance

#### Board oversight

In 2024, Eurolux Universal Bank AG’s board comprised 10 members, with 68.5% classified as independent directors. Climate-related matters featured on the agenda of 72.6% of board meetings, which were held eight times during the year. The board receives climate risk reporting on a semi-annual basis, supporting its oversight responsibilities.

The board’s oversight of climate-related risks and opportunities includes review and endorsement of the bank’s transition plans, climate sc

In [13]:

# ============================================================
# STRATEGY SECTION GENERATOR
# IFRS S1/S2 strategy | uses same Azure REST helper functions
# ============================================================
# This section is added after Governance and reuses:
# - payload
# - bank_name
# - call_llm()
# - call_llm_json()
# - _is_present(), _safe_int(), _normalise_text()


def _safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        # handle NaN from JSON payloads
        if isinstance(value, float) and value != value:
            return default
        return float(value)
    except Exception:
        return default


def _json_clean(obj):
    """Make dictionaries/lists safe for JSON prompt dumps by replacing NaN with None."""
    if isinstance(obj, dict):
        return {k: _json_clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_json_clean(v) for v in obj]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def _pick_latest(records: list[dict], year: int = 2024) -> dict:
    for r in records:
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year:
            return r
    return records[-1] if records else {}


print("Strategy helper functions ready")


Strategy helper functions ready


In [14]:
# ── STRATEGY EVIDENCE EXTRACTOR ─────────────────────────────
# Pulls only strategy-relevant fields from the payload.
# Updated to support the enhanced strategy payload:
# - value_chain_map and value_chain_summary
# - climate_opportunities and opportunity_summary
# - business_model_impacts
# - strategy_tradeoff_decisions
# - portfolio_exposure_summary
# - strategy_evidence_boundaries


def _load_strategy_payload_if_available(default_payload: dict) -> dict:
    """Use an enhanced strategy-only payload if present; otherwise reuse the main payload."""
    candidate_paths = [
        Path("Data/payload_BANK01_strategy_enhanced.json"),
        Path("payload_BANK01_strategy_enhanced.json"),
        Path("Data/payload_BANK01_strategy.json"),
        Path("payload_BANK01_strategy.json"),
        Path.cwd() / "Data" / "payload_BANK01_strategy_enhanced.json",
        Path.cwd() / "payload_BANK01_strategy_enhanced.json",
        Path.cwd() / "Data" / "payload_BANK01_strategy.json",
        Path.cwd() / "payload_BANK01_strategy.json",
    ]
    strategy_path = next((p for p in candidate_paths if p.exists()), None)
    if strategy_path:
        with open(strategy_path, "r", encoding="utf-8") as f:
            print(f"Loaded strategy-specific payload from: {strategy_path}")
            return json.load(f)
    print("No strategy-specific payload found; using main payload for strategy.")
    return default_payload


def summarize_scenarios(payload: dict, year: int = 2024) -> dict:
    scenarios = [s for s in payload.get("climate_scenarios", []) if isinstance(s, dict)]

    if not scenarios:
        return {
            "available": False,
            "message": "No climate_scenarios records available.",
        }

    by_type = {}
    by_horizon = {}
    scenario_names = sorted({str(s.get("scenario_name")) for s in scenarios if _is_present(s.get("scenario_name"))})
    frameworks = sorted({str(s.get("framework")) for s in scenarios if _is_present(s.get("framework"))})
    horizon_years = sorted({_safe_int(s.get("horizon_year")) for s in scenarios if _safe_int(s.get("horizon_year"))})

    for s in scenarios:
        st = str(s.get("scenario_type") or "unknown")
        hz = str(s.get("horizon") or "unknown")
        by_type.setdefault(st, 0)
        by_type[st] += 1
        by_horizon.setdefault(hz, 0)
        by_horizon[hz] += 1

    def max_by(field):
        candidates = [s for s in scenarios if _is_present(s.get(field))]
        if not candidates:
            return None
        return max(candidates, key=lambda x: _safe_float(x.get(field)))

    max_physical = max_by("physical_risk_loss_pct_capital")
    max_transition = max_by("transition_risk_loss_pct_capital")
    max_revenue = max_by("revenue_at_risk_meur")
    max_stranded = max_by("stranded_assets_estimate_meur")

    scenario_examples = []
    for s in scenarios[:10]:
        scenario_examples.append({
            "scenario_id": s.get("scenario_id"),
            "scenario_name": s.get("scenario_name"),
            "scenario_type": s.get("scenario_type"),
            "horizon": s.get("horizon"),
            "horizon_year": s.get("horizon_year"),
            "temperature_outcome_c": s.get("temperature_outcome_c"),
            "physical_risk_loss_pct_capital": s.get("physical_risk_loss_pct_capital"),
            "transition_risk_loss_pct_capital": s.get("transition_risk_loss_pct_capital"),
            "revenue_at_risk_meur": s.get("revenue_at_risk_meur"),
            "stranded_assets_estimate_meur": s.get("stranded_assets_estimate_meur"),
            "high_risk_exposure_pct": s.get("high_risk_exposure_pct"),
            "carbon_price_assumption_eur_per_tco2e": s.get("carbon_price_assumption_eur_per_tco2e"),
            "methodology_notes": s.get("methodology_notes"),
            "resilience_assessment": s.get("resilience_assessment"),
        })

    return _json_clean({
        "available": True,
        "scenario_count": len(scenarios),
        "frameworks": frameworks,
        "scenario_names": scenario_names,
        "scenario_counts_by_type": by_type,
        "scenario_counts_by_horizon": by_horizon,
        "horizon_years": horizon_years,
        "max_physical_loss_scenario": max_physical,
        "max_transition_loss_scenario": max_transition,
        "max_revenue_at_risk_scenario": max_revenue,
        "max_stranded_assets_scenario": max_stranded,
        "scenario_examples": scenario_examples,
    })


def summarize_strategy_risks(payload: dict, year: int = 2024) -> dict:
    risks = [
        r for r in payload.get("climate_risk_register", [])
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
    ]

    physical = [r for r in risks if "physical" in str(r.get("risk_category", "")).lower()]
    transition = [r for r in risks if "transition" in str(r.get("risk_category", "")).lower()]

    def compact(r):
        return {
            "risk_id": r.get("risk_id"),
            "risk_name": r.get("risk_name"),
            "risk_category": r.get("risk_category"),
            "time_horizon": r.get("time_horizon"),
            "risk_rating": r.get("risk_rating"),
            "financial_impact_meur": r.get("financial_impact_meur"),
            "scenario_analysis_link": r.get("scenario_analysis_link"),
            "mitigation_actions": r.get("mitigation_actions"),
        }

    rating_priority = {"critical": 4, "high": 3, "medium": 2, "low": 1}
    risks_sorted = sorted(
        risks,
        key=lambda r: (
            rating_priority.get(str(r.get("risk_rating", "")).lower(), 0),
            _safe_float(r.get("financial_impact_meur"))
        ),
        reverse=True,
    )

    return _json_clean({
        "risk_count": len(risks),
        "physical_risk_count": len(physical),
        "transition_risk_count": len(transition),
        "risk_categories": sorted({str(r.get("risk_category")) for r in risks if _is_present(r.get("risk_category"))}),
        "time_horizons": sorted({str(r.get("time_horizon")) for r in risks if _is_present(r.get("time_horizon"))}),
        "risk_ratings": sorted({str(r.get("risk_rating")) for r in risks if _is_present(r.get("risk_rating"))}),
        "top_risk_examples": [compact(r) for r in risks_sorted[:8]],
    })


def summarize_physical_exposures(payload: dict) -> dict:
    exposures = [x for x in payload.get("physical_risk_exposures", []) if isinstance(x, dict)]
    if not exposures:
        return {"available": False}

    high_risk = [x for x in exposures if x.get("high_risk_flag") is True]
    total_exposure = sum(_safe_float(x.get("exposure_amount_meur")) for x in exposures)
    total_impact = sum(_safe_float(x.get("financial_impact_meur")) for x in exposures)
    high_risk_exposure = sum(_safe_float(x.get("exposure_amount_meur")) for x in high_risk)

    hazard_counts = {}
    country_exposure = {}
    for x in exposures:
        hazard = str(x.get("hazard_type") or "unknown")
        hazard_counts[hazard] = hazard_counts.get(hazard, 0) + 1
        country = str(x.get("country") or "unknown")
        country_exposure[country] = country_exposure.get(country, 0.0) + _safe_float(x.get("exposure_amount_meur"))

    top_countries = sorted(country_exposure.items(), key=lambda kv: kv[1], reverse=True)[:5]

    return _json_clean({
        "available": True,
        "record_count": len(exposures),
        "high_risk_record_count": len(high_risk),
        "total_physical_exposure_meur": round(total_exposure, 2),
        "high_risk_physical_exposure_meur": round(high_risk_exposure, 2),
        "total_financial_impact_meur": round(total_impact, 2),
        "hazard_counts": hazard_counts,
        "top_countries_by_exposure_meur": top_countries,
    })


def summarize_value_chain(payload: dict) -> dict:
    nodes = [v for v in payload.get("value_chain_map", []) if isinstance(v, dict)]
    supplied_summary = payload.get("value_chain_summary")
    if supplied_summary:
        supplied_summary = dict(supplied_summary)
    else:
        supplied_summary = {}

    if not nodes and not supplied_summary:
        return {"available": False}

    material = [v for v in nodes if v.get("materiality_flag") is True]
    quantified = [v for v in nodes if _is_present(v.get("financial_exposure_meur"))]

    top_nodes = sorted(
        material,
        key=lambda v: _safe_float(v.get("financial_exposure_meur")),
        reverse=True,
    )[:8]

    compact_nodes = [{
        "node_name": v.get("node_name"),
        "node_type": v.get("node_type"),
        "position": v.get("upstream_downstream"),
        "materiality_flag": v.get("materiality_flag"),
        "climate_exposure_type": v.get("climate_exposure_type"),
        "financial_exposure_meur": v.get("financial_exposure_meur"),
        "scope3_category": v.get("scope3_category"),
        "description": v.get("climate_risk_description"),
    } for v in top_nodes]

    return _json_clean({
        "available": True,
        "total_nodes": supplied_summary.get("total_nodes", len(nodes)),
        "material_nodes": supplied_summary.get("material_nodes", len(material)),
        "quantified_nodes": supplied_summary.get("quantified_nodes", len(quantified)),
        "nodes_by_type": supplied_summary.get("nodes_by_type"),
        "nodes_by_position": supplied_summary.get("nodes_by_position"),
        "top_material_nodes": compact_nodes or supplied_summary.get("top_material_financing_counterparties", []),
        "qualitative_nodes_note": supplied_summary.get("qualitative_nodes_note"),
    })


def summarize_opportunities(payload: dict) -> dict:
    supplied = payload.get("opportunity_summary")
    if supplied:
        return _json_clean({"available": True, **supplied})

    opportunities = [o for o in payload.get("climate_opportunities", []) if isinstance(o, dict)]
    if not opportunities:
        return {"available": False}

    return _json_clean({
        "available": True,
        "opportunity_count": len(opportunities),
        "total_estimated_revenue_impact_meur": round(sum(_safe_float(o.get("estimated_revenue_impact_meur")) for o in opportunities), 2),
        "opportunities": [{
            "opportunity_type": o.get("opportunity_type"),
            "category": o.get("category"),
            "description": o.get("description"),
            "estimated_revenue_impact_meur": o.get("estimated_revenue_impact_meur"),
            "time_horizon": o.get("time_horizon"),
            "confidence_level": o.get("confidence_level"),
        } for o in opportunities],
        "note": "Revenue impacts are estimates with stated confidence levels, not guaranteed revenue.",
    })


def extract_strategy_evidence(payload: dict, year: int = 2024) -> dict:
    strategy_payload = _load_strategy_payload_if_available(payload)
    bank = strategy_payload.get("bank", {})
    reporting_kpis = strategy_payload.get("reporting_kpis", {})
    fin_latest = _pick_latest(strategy_payload.get("financial_summary", []), year)
    fin_trend = [f for f in strategy_payload.get("financial_summary", []) if isinstance(f, dict)]

    portfolio_exposure_summary = strategy_payload.get("portfolio_exposure_summary") or {
        "total_loans_meur": reporting_kpis.get("total_loans_2024_meur"),
        "green_loans_pct": reporting_kpis.get("green_loans_pct_2024"),
        "high_carbon_sector_exposure_pct": reporting_kpis.get("high_carbon_sector_exposure_pct"),
        "high_carbon_sector_exposure_meur": reporting_kpis.get("high_carbon_sector_exposure_meur"),
        "fossil_fuel_exposure_pct": reporting_kpis.get("fossil_fuel_exposure_pct"),
        "fossil_fuel_exposure_meur": reporting_kpis.get("fossil_fuel_exposure_meur"),
        "financed_emissions_tco2e": reporting_kpis.get("financed_emissions_2024_tco2e"),
        "carbon_intensity_tco2e_per_meur": reporting_kpis.get("carbon_intensity_2024_tco2e_per_meur"),
    }

    boundaries = strategy_payload.get("strategy_evidence_boundaries") or {}
    evidence = {
        "bank": {
            "bank_id": bank.get("bank_id"),
            "bank_name": bank.get("bank_name"),
            "country": bank.get("country"),
            "archetype": bank.get("archetype"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "total_loans_meur": bank.get("total_loans_meur"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "reporting_year": year,
        "financial_summary_2024": fin_latest,
        "financial_summary_trend": fin_trend,
        "strategy_kpis": {
            "green_loans_pct_2024": reporting_kpis.get("green_loans_pct_2024"),
            "high_carbon_sector_exposure_pct": reporting_kpis.get("high_carbon_sector_exposure_pct"),
            "fossil_fuel_exposure_pct": reporting_kpis.get("fossil_fuel_exposure_pct"),
            "high_carbon_sector_exposure_meur": reporting_kpis.get("high_carbon_sector_exposure_meur"),
            "fossil_fuel_exposure_meur": reporting_kpis.get("fossil_fuel_exposure_meur"),
            "climate_capex_2024_meur": reporting_kpis.get("climate_capex_2024_meur"),
            "climate_opex_2024_meur": reporting_kpis.get("climate_opex_2024_meur"),
            "financed_emissions_2024_tco2e": reporting_kpis.get("financed_emissions_2024_tco2e"),
            "carbon_intensity_2024_tco2e_per_meur": reporting_kpis.get("carbon_intensity_2024_tco2e_per_meur"),
        },
        "portfolio_exposure_summary": portfolio_exposure_summary,
        "targets": reporting_kpis.get("target_summary", []),
        "risk_register_summary": summarize_strategy_risks(strategy_payload, year=year),
        "scenario_summary": summarize_scenarios(strategy_payload, year=year),
        "physical_exposure_summary": summarize_physical_exposures(strategy_payload),
        "value_chain_summary": summarize_value_chain(strategy_payload),
        "opportunity_summary": summarize_opportunities(strategy_payload),
        "business_model_impacts": strategy_payload.get("business_model_impacts", []),
        "strategy_tradeoff_decisions": strategy_payload.get("strategy_tradeoff_decisions", []),
        "evidence_boundaries": {
            "scenario_outputs_are_modelled_estimates": True,
            "opportunity_impacts_are_estimates_not_guaranteed": True,
            "do_not_claim_overall_resilience_without_scenario_context": True,
            "do_not_claim_bank_wide_paris_alignment": True,
            "formal_quantified_tradeoff_analysis_available": boundaries.get("formal_quantified_tradeoff_analysis_available", False),
            "tradeoff_evidence_available_qualitative": boundaries.get("tradeoff_evidence_available_qualitative", bool(strategy_payload.get("strategy_tradeoff_decisions"))),
            "value_chain_detail_available": bool(strategy_payload.get("value_chain_map") or strategy_payload.get("value_chain_summary")),
            "business_model_detail_available": bool(strategy_payload.get("business_model_impacts")),
            "high_carbon_and_fossil_exposure_available": _is_present(reporting_kpis.get("high_carbon_sector_exposure_pct")) and _is_present(reporting_kpis.get("fossil_fuel_exposure_pct")),
        },
    }

    return _json_clean(evidence)


strategy_evidence = extract_strategy_evidence(payload, year=2024)
print("Strategy evidence extracted")
print(json.dumps({
    "bank": strategy_evidence["bank"],
    "scenario_count": strategy_evidence["scenario_summary"].get("scenario_count"),
    "risk_count": strategy_evidence["risk_register_summary"].get("risk_count"),
    "value_chain_available": strategy_evidence["value_chain_summary"].get("available"),
    "opportunities_available": strategy_evidence["opportunity_summary"].get("available"),
    "business_model_impacts": len(strategy_evidence.get("business_model_impacts", [])),
    "strategy_tradeoffs": len(strategy_evidence.get("strategy_tradeoff_decisions", [])),
    "high_carbon_pct": strategy_evidence["strategy_kpis"].get("high_carbon_sector_exposure_pct"),
    "fossil_fuel_pct": strategy_evidence["strategy_kpis"].get("fossil_fuel_exposure_pct"),
}, indent=2, ensure_ascii=False))


No strategy-specific payload found; using main payload for strategy.
Strategy evidence extracted
{
  "bank": {
    "bank_id": "BANK01",
    "bank_name": "Eurolux Universal Bank AG",
    "country": "DE",
    "archetype": "large_universal",
    "total_assets_meur": 850000,
    "total_loans_meur": 31150.64,
    "regulatory_regime": "CSRD"
  },
  "scenario_count": 18,
  "risk_count": 8,
  "value_chain_available": true,
  "opportunities_available": false,
  "business_model_impacts": 0,
  "strategy_tradeoffs": 0,
  "high_carbon_pct": null,
  "fossil_fuel_pct": null
}


In [15]:

# ── STRATEGY STATE DEFINITION ───────────────────────────────
class StrategyState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict


In [16]:

# ── STRATEGY REQUIREMENTS ───────────────────────────────────
# IFRS references are used internally only. Final text must NOT show IFRS paragraph references.

STRATEGY_REQUIREMENTS = """
STRICT STRATEGY DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Final section title and headings must be exactly:
### Strategy
#### Climate-related risks and opportunities
#### Effects on business model and value chain
#### Effects on strategy and decision-making
#### Financial effects and resource allocation
#### Climate resilience and scenario analysis
#### Strategy limitations and evidence boundaries

Coverage requirements:
1. Identify climate-related risks and opportunities using evidence from the risk register, KPIs, targets and scenario analysis.
2. Distinguish physical risks from transition risks.
3. Cover short, medium and long-term horizons where scenario or risk-register evidence supports them.
4. Explain effects on business model and value chain using value_chain_map/value_chain_summary and business_model_impacts where available. If only partial financial quantification exists, state that boundary.
5. Explain effects on strategy and decision-making using evidence such as strategy_tradeoff_decisions, transition plan, net-zero target revision, climate scenario methodology, carbon credit budget, climate capex/opex, green loans and exposure metrics.
6. Include financial effects and resource allocation using numeric evidence: high-carbon exposure, fossil fuel exposure, climate capex, climate opex, revenue at risk, stranded assets, physical/transition loss percentages or financed emissions if relevant.
7. Include climate resilience and scenario analysis. Explain scenario families/types, frameworks, time horizons and key assumptions.
8. Do not claim the bank is resilient in general. Only describe resilience within the boundaries of the scenario evidence.
9. Do not claim Paris alignment unless directly supported by the evidence. If a scenario or target has a flag, explain it cautiously.
10. Do not overstate modelled estimates as actual losses.
11. Do not include visible IFRS paragraph references or bracketed IFRS tags in the final output.
12. If business model, value-chain, opportunity, trade-off, high-carbon exposure or fossil-fuel exposure evidence is present, use it. Do not state it is unavailable. If evidence is partial, disclose the boundary.
""".strip()


In [17]:

# ── STRATEGY WRITER / JUDGE PROMPTS ─────────────────────────
STRATEGY_WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Strategy section of an IFRS S1/S2 aligned climate disclosure report for a commercial bank.

WRITING STANDARDS:
- Formal, third-person professional disclosure style.
- Evidence-based and audit-friendly.
- Do not use markdown tables.
- Do not include visible IFRS paragraph references, paragraph numbers, or bracketed IFRS tags.
- Do not invent strategy, value-chain, financial effect, or resilience claims.
- If evidence is missing, state the limitation in report-style language using "available evidence" or "source data". Avoid saying "payload" in final text.

STRICT RULES:
- Distinguish physical risk from transition risk.
- Distinguish modelled scenario outputs from actual financial impacts.
- Do not claim the bank is resilient overall. You may say the scenario evidence supports assessment within specified scenario boundaries.
- Do not claim Paris alignment unless directly supported by target or scenario flags, and even then use cautious wording.
- Avoid unsupported words: "ensures", "guarantees", "fully resilient", "compliant", "fully aligned", "proves".
- Keep board decision details concise if mentioned; Strategy is not Governance.
- Use value_chain_summary/value_chain_map when available; do not state value-chain evidence is unavailable when those fields are present.
- Use business_model_impacts when available to explain actual business model channels.
- Use climate_opportunities/opportunity_summary when available, but describe revenue impacts as estimates with confidence levels, not guaranteed revenue.
- Use high-carbon sector exposure and fossil-fuel exposure values when present; never say those data are unavailable if they exist in evidence.
- Use strategy_tradeoff_decisions if available, while noting whether quantified trade-off analysis is unavailable.
""".strip()


def build_strategy_writer_prompt(evidence: dict, judge_feedback: str | None = None) -> str:
    feedback_block = ""
    if judge_feedback:
        feedback_block = f"""
PREVIOUS JUDGE FEEDBACK TO FIX:
{judge_feedback}
""".strip()

    return f"""
{STRATEGY_REQUIREMENTS}

BANK:
{evidence['bank']['bank_name']} ({evidence['bank']['bank_id']})

EVIDENCE JSON:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

{feedback_block}

Write the final Strategy section only.
""".strip()


STRATEGY_JUDGE_SYSTEM = """
You are a strict IFRS S1/S2 Strategy disclosure reviewer and ESG audit specialist.
Your role is to identify genuine gaps, not to reward fluent writing.

Return ONLY valid JSON with this schema:
{
  "overall_score": 0-10,
  "evidence_support_score": 0-10,
  "ifrs_alignment_score": 0-10,
  "specificity_score": 0-10,
  "hallucination_risk": "low" | "medium" | "high",
  "approved": true | false,
  "checklist": {
    "all_six_subsections_present": true | false,
    "no_visible_ifrs_refs": true | false,
    "physical_transition_distinct": true | false,
    "time_horizons_covered": true | false,
    "business_model_effects_or_limitation": true | false,
    "value_chain_effects_or_limitation": true | false,
    "value_chain_data_used_if_available": true | false,
    "strategy_decision_making_covered": true | false,
    "financial_effects_covered": true | false,
    "high_carbon_fossil_exposure_used_if_available": true | false,
    "resource_allocation_covered": true | false,
    "opportunities_used_if_available": true | false,
    "tradeoffs_used_if_available": true | false,
    "scenario_analysis_covered": true | false,
    "scenario_assumptions_covered": true | false,
    "resilience_not_overclaimed": true | false,
    "paris_alignment_not_overclaimed": true | false,
    "modelled_estimates_not_overstated": true | false,
    "evidence_boundaries_present": true | false,
    "no_unsupported_strong_claims": true | false,
    "false_count": 0
  },
  "main_issues": [],
  "required_fixes": []
}

SCORING RULES:
- 9-10 only if strategy is directly evidenced and no material limitation is needed.
- 8 if strong but one material element is addressed only through limitation.
- 7 if two or more material elements are addressed through limitations, but no hallucination.
- 6 or lower if it overclaims resilience, Paris alignment, financial effects, or uses unsupported strong claims.
- A limitation statement reduces hallucination risk but does not count as full completeness.
- If evidence contains value_chain_summary/value_chain_map but the draft says value-chain evidence is unavailable, cap the score at 6.
- If evidence contains high-carbon and fossil-fuel exposure values but the draft says these data are unavailable, cap the score at 6.
- If evidence contains climate_opportunities but the draft does not mention concrete opportunities, cap the score at 7.
- If evidence contains strategy_tradeoff_decisions but the draft says no trade-off evidence is available, cap the score at 7.
""".strip()


def build_strategy_judge_prompt(draft: str, evidence: dict) -> str:
    return f"""
Evaluate the Strategy section against the strict requirements.

EVIDENCE JSON:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

DRAFT:
{draft}

Return only valid JSON.
""".strip()

print("Strategy prompts ready")


Strategy prompts ready


In [18]:

# ── STRATEGY DETERMINISTIC RULE CHECKS ──────────────────────

def check_no_visible_ifrs_refs(text: str) -> bool:
    patterns = [
        r"\bIFRS\s+S[12]\b",
        r"\bIFRS\b",
        r"§\s*\d+",
        r"\[\s*IFRS",
    ]
    return not any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)


def check_strategy_headings(draft: str) -> bool:
    required = [
        "### Strategy",
        "#### Climate-related risks and opportunities",
        "#### Effects on business model and value chain",
        "#### Effects on strategy and decision-making",
        "#### Financial effects and resource allocation",
        "#### Climate resilience and scenario analysis",
        "#### Strategy limitations and evidence boundaries",
    ]
    lower = draft.lower()
    return all(h.lower() in lower for h in required)


def has_physical_and_transition(text: str) -> bool:
    t = text.lower()
    return "physical" in t and "transition" in t


def has_scenario_boundaries(text: str) -> bool:
    t = text.lower()
    boundary_terms = ["scenario", "modelled", "assumption", "available evidence", "source data", "within the", "boundary", "does not"]
    return "scenario" in t and sum(term in t for term in boundary_terms) >= 3


def resilience_not_overclaimed(text: str) -> bool:
    t = text.lower()
    banned = [
        "fully resilient",
        "is resilient across all scenarios",
        "proves resilience",
        "guarantees resilience",
        "ensures resilience",
    ]
    if any(b in t for b in banned):
        return False
    # If resilience is mentioned, it must be linked to scenario evidence / boundaries.
    if "resilien" in t:
        return any(term in t for term in ["scenario", "modelled", "available evidence", "within", "under"])
    return True


def paris_alignment_not_overclaimed(text: str) -> bool:
    t = text.lower()
    banned = ["fully paris-aligned", "is paris-aligned", "paris aligned portfolio", "guarantees paris alignment"]
    return not any(b in t for b in banned)


def modelled_estimates_not_overstated(text: str) -> bool:
    t = text.lower()
    bad_phrases = [
        "actual loss of",
        "actual losses of",
        "incurred loss of",
        "realized loss of",
        "realised loss of",
    ]
    return not any(p in t for p in bad_phrases)


def strategy_rule_check(draft: str) -> list[str]:
    failures = []
    if not check_strategy_headings(draft):
        failures.append("missing_strategy_headings")
    if not check_no_visible_ifrs_refs(draft):
        failures.append("visible_ifrs_references")
    if not has_physical_and_transition(draft):
        failures.append("physical_transition_not_distinguished")
    if not has_scenario_boundaries(draft):
        failures.append("missing_scenario_boundaries")
    if not resilience_not_overclaimed(draft):
        failures.append("resilience_overclaimed")
    if not paris_alignment_not_overclaimed(draft):
        failures.append("paris_alignment_overclaimed")
    if not modelled_estimates_not_overstated(draft):
        failures.append("modelled_estimates_overstated")
    return failures


def strict_strategy_score(raw_score: int, checklist: dict) -> tuple[int, str]:
    score = int(raw_score or 0)
    cap_reasons = []

    limitation_keys = [
        "business_model_effects_or_limitation",
        "value_chain_effects_or_limitation",
        "evidence_boundaries_present",
    ]
    # These are not exactly limitation flags, but if evidence boundaries are present while direct effects are weak,
    # the judge prompt should reflect incompleteness in the score. Use caps for hard failures below.

    hard_failure_caps = [
        (not checklist.get("all_six_subsections_present", True), 6, "required subsection missing"),
        (not checklist.get("no_visible_ifrs_refs", True), 7, "visible IFRS references"),
        (not checklist.get("physical_transition_distinct", True), 7, "physical and transition risks not distinguished"),
        (not checklist.get("scenario_analysis_covered", True), 7, "scenario analysis not covered"),
        (not checklist.get("resilience_not_overclaimed", True), 6, "resilience overclaimed"),
        (not checklist.get("paris_alignment_not_overclaimed", True), 6, "Paris alignment overclaimed"),
        (not checklist.get("modelled_estimates_not_overstated", True), 6, "modelled estimates overstated"),
        (not checklist.get("no_unsupported_strong_claims", True), 6, "unsupported strong claims"),
    ]

    for condition, cap, reason in hard_failure_caps:
        if condition:
            if score > cap:
                cap_reasons.append(f"Capped at {cap}: {reason}")
            score = min(score, cap)

    return score, "; ".join(cap_reasons) if cap_reasons else "No cap applied"

print("Strategy rule checks ready")


Strategy rule checks ready


In [19]:

# ── STRATEGY LANGGRAPH NODES ────────────────────────────────

def strategy_writer_node(state: StrategyState) -> StrategyState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_strategy_writer_prompt(state["evidence"], judge_feedback=feedback)
    draft = call_llm(
        system_prompt=STRATEGY_WRITER_SYSTEM,
        user_prompt=prompt,
        temperature=0.2,
    )

    print(f"\n{'='*50}")
    print(f"STRATEGY WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {**state, "draft": draft.strip(), "status": "judging"}


def strategy_judge_node(state: StrategyState) -> StrategyState:
    draft = state["draft"]
    rule_failures = strategy_rule_check(draft)

    if rule_failures:
        judge_result = {
            "overall_score": 4,
            "evidence_support_score": 5,
            "ifrs_alignment_score": 4,
            "specificity_score": 5,
            "hallucination_risk": "medium",
            "approved": False,
            "checklist": {
                "all_six_subsections_present": check_strategy_headings(draft),
                "no_visible_ifrs_refs": check_no_visible_ifrs_refs(draft),
                "physical_transition_distinct": has_physical_and_transition(draft),
                "scenario_analysis_covered": "scenario" in draft.lower(),
                "resilience_not_overclaimed": resilience_not_overclaimed(draft),
                "paris_alignment_not_overclaimed": paris_alignment_not_overclaimed(draft),
                "modelled_estimates_not_overstated": modelled_estimates_not_overstated(draft),
                "false_count": len(rule_failures),
            },
            "main_issues": [f"Fix hard fail: {f}" for f in rule_failures],
            "required_fixes": [f"Fix hard fail: {f}" for f in rule_failures],
            "rule_check_override": True,
        }
    else:
        prompt = build_strategy_judge_prompt(draft, state["evidence"])
        judge_result = call_llm_json(
            system_prompt=STRATEGY_JUDGE_SYSTEM,
            user_prompt=prompt,
        )

        checklist = judge_result.get("checklist", {})
        false_count = sum(1 for k, v in checklist.items() if v is False and k != "false_count")
        checklist["false_count"] = false_count
        judge_result["checklist"] = checklist

        raw_score = int(judge_result.get("overall_score", 0))
        final_score, cap_reason = strict_strategy_score(raw_score, checklist)
        judge_result["raw_score_before_caps"] = raw_score
        judge_result["overall_score"] = final_score
        judge_result["score_cap_reason"] = cap_reason

        # Approval logic
        if final_score >= 8 and false_count == 0:
            judge_result["approved"] = True
            judge_result["approval_status"] = "approved"
        elif final_score >= 7 and false_count <= 2:
            judge_result["approved"] = True
            judge_result["approval_status"] = "approved_with_limitations"
        else:
            judge_result["approved"] = False
            judge_result["approval_status"] = "revision_required"

    print("\nSTRATEGY JUDGE RESULT")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))

    approved = bool(judge_result.get("approved"))
    status = "approved" if approved else "revising"

    return {**state, "judge_result": judge_result, "status": status}


def strategy_reviser_node(state: StrategyState) -> StrategyState:
    if state["revision_count"] >= state["max_revisions"]:
        print("Max strategy revisions reached. Marking as failed.")
        return {**state, "status": "failed", "final_section": state["draft"]}

    return {**state, "revision_count": state["revision_count"] + 1, "status": "drafting"}


def strategy_finalize_node(state: StrategyState) -> StrategyState:
    return {**state, "final_section": state["draft"], "status": "approved"}


def strategy_should_continue(state: StrategyState) -> str:
    if state["status"] == "approved":
        return "finalize"
    if state["status"] == "failed":
        return END
    return "reviser"

print("Strategy nodes ready")


Strategy nodes ready


In [20]:

# ── BUILD AND COMPILE STRATEGY GRAPH ────────────────────────
strategy_builder = StateGraph(StrategyState)

strategy_builder.add_node("writer",   strategy_writer_node)
strategy_builder.add_node("judge",    strategy_judge_node)
strategy_builder.add_node("reviser",  strategy_reviser_node)
strategy_builder.add_node("finalize", strategy_finalize_node)

strategy_builder.add_edge(START, "writer")
strategy_builder.add_edge("writer", "judge")
strategy_builder.add_conditional_edges(
    "judge",
    strategy_should_continue,
    {
        "finalize": "finalize",
        "reviser": "reviser",
        END: END,
    }
)
strategy_builder.add_edge("reviser", "writer")
strategy_builder.add_edge("finalize", END)

strategy_graph = strategy_builder.compile()
print("Strategy graph compiled")


Strategy graph compiled


In [21]:

# ── RUN STRATEGY SECTION ────────────────────────────────────
strategy_initial_state: StrategyState = {
    "bank_name":      bank_name,
    "evidence":       strategy_evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions":  0,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {},
}

print(f"Starting strategy generation for: {bank_name}\n")
strategy_result = strategy_graph.invoke(strategy_initial_state)


Starting strategy generation for: Eurolux Universal Bank AG


STRATEGY WRITER (initial draft)
Draft length: 1145 words

STRATEGY JUDGE RESULT
{
  "overall_score": 8,
  "evidence_support_score": 9,
  "ifrs_alignment_score": 8,
  "specificity_score": 8,
  "hallucination_risk": "low",
  "approved": false,
  "checklist": {
    "all_six_subsections_present": true,
    "no_visible_ifrs_refs": true,
    "physical_transition_distinct": true,
    "time_horizons_covered": true,
    "business_model_effects_or_limitation": false,
    "value_chain_effects_or_limitation": true,
    "value_chain_data_used_if_available": true,
    "strategy_decision_making_covered": true,
    "financial_effects_covered": true,
    "high_carbon_fossil_exposure_used_if_available": false,
    "resource_allocation_covered": true,
    "opportunities_used_if_available": false,
    "tradeoffs_used_if_available": false,
    "scenario_analysis_covered": true,
    "scenario_assumptions_covered": true,
    "resilience_not_overcl

KeyboardInterrupt: 

In [ ]:

# ── STRATEGY OUTPUT ─────────────────────────────────────────
print("\n" + "="*60)
print("FINAL STRATEGY JUDGE RESULT")
print("="*60)
print(json.dumps(strategy_result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("STRATEGY SECTION")
print("="*60)
print(strategy_result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "strategy_BANK01.md", "w", encoding="utf-8") as f:
    f.write(strategy_result["final_section"])

with open(output_dir / "strategy_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "strategy",
        "status":         strategy_result["status"],
        "approval_status": strategy_result["judge_result"].get("approval_status"),
        "raw_score":      strategy_result["judge_result"].get("raw_score_before_caps"),
        "final_score":    strategy_result["judge_result"].get("overall_score"),
        "score_cap_reason": strategy_result["judge_result"].get("score_cap_reason"),
        "revisions":      strategy_result["revision_count"],
        "approved":       strategy_result["judge_result"].get("approved"),
        "checklist":      strategy_result["judge_result"].get("checklist"),
        "issues":         strategy_result["judge_result"].get("main_issues"),
    }, f, indent=2, ensure_ascii=False)

print("\nSaved to outputs/strategy_BANK01.md")
